# BUG 04 — La transformación intragrupos destruye el 94 % de la muestra

**Unidad 4.c** · Acompaña a `Clase_04_DatosPanel` · Notas: cap. 5

> **Este cuaderno contiene un error deliberado.** No lo corrijas todavía: ejecútalo,
> observa la salida y sigue las tareas del final. Las soluciones están en
> [`SOLUCIONES.md`](SOLUCIONES.md), que conviene no abrir antes de intentarlo.

Implementamos a mano la **transformación intragrupos** —restar a cada variable la media
de su grupo— para estimar efectos fijos sobre el panel de salarios. El cuaderno corre sin
errores y estima una regresión.

Este error es de los más frecuentes en pandas y de los más difíciles de ver, porque la
operación que lo produce es perfectamente válida: sólo alinea por la etiqueta equivocada.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

panel = pd.read_csv("../../Clase_04_DatosPanel/wage_panel.csv")

print(f"Observaciones: {len(panel)}")
print(f"Individuos (nr): {panel['nr'].nunique()}")
print(f"Años: {panel['year'].min()}-{panel['year'].max()}")
panel.head()

Observaciones: 4360
Individuos (nr): 545
Años: 1980-1987


,nr,year,black,exper,hisp,hours,married,educ,union,lwage,expersq,occupation
0,13,1980,0,1,0,2672,0,14,0,1.197540,1,9
1,13,1981,0,2,0,2320,0,14,1,1.853060,4,9
2,13,1982,0,3,0,2940,0,14,0,1.344462,9,9
3,13,1983,0,4,0,2960,0,14,0,1.433213,16,9
4,13,1984,0,5,0,3071,0,14,0,1.568125,25,5


In [2]:
VARIABLES = ["lwage", "exper", "expersq", "union", "married"]

# Transformación intragrupos: a cada variable se le resta la media de su individuo.
centradas = {}
for variable in VARIABLES:
    media_por_individuo = panel.groupby("nr")[variable].mean()
    centradas[variable + "_c"] = panel[variable] - media_por_individuo

datos = pd.DataFrame(centradas)

print(f"Renglones después de centrar: {len(datos)}\n")
print("Valores no nulos por variable centrada:")
for columna in datos.columns:
    validos = datos[columna].notna().sum()
    print(f"  {columna:12s} {validos:5d} de {len(datos)}  ({100 * validos / len(datos):.1f} %)")

Renglones después de centrar: 4639

Valores no nulos por variable centrada:
  lwage_c        266 de 4639  (5.7 %)
  exper_c        266 de 4639  (5.7 %)
  expersq_c      266 de 4639  (5.7 %)
  union_c        266 de 4639  (5.7 %)
  married_c      266 de 4639  (5.7 %)


De 4,360 observaciones sobreviven 266. No hay ningún mensaje de error.

La regresión corre igual:

In [3]:
formula = "lwage_c ~ exper_c + expersq_c + union_c + married_c - 1"
modelo = smf.ols(formula, data=datos).fit()

print(f"N efectivo = {int(modelo.nobs)}   R^2 = {modelo.rsquared:.4f}\n")
pd.DataFrame({"coef": modelo.params, "ee": modelo.bse}).round(4)

N efectivo = 266   R^2 = 0.1595



,coef,ee
exper_c,0.1142,0.0517
expersq_c,-0.0076,0.0035
union_c,0.2467,0.0619
married_c,0.2302,0.0587


## La prueba que lo delata

Una variable centrada por grupo tiene, **por construcción**, media cero dentro de cada
grupo. Esta comprobación no requiere conocer la respuesta correcta: se deriva de la
definición de la transformación.

In [4]:
comprobacion = pd.DataFrame({"nr": panel["nr"], "lwage_c": datos["lwage_c"]})
medias = comprobacion.groupby("nr")["lwage_c"].mean().dropna()

print(f"Grupos evaluables: {len(medias)} de {panel['nr'].nunique()}")
print(f"Máxima desviación de cero: {medias.abs().max():.6f}")
print()
print("Debería ser del orden de 1e-15, no mayor.")

Grupos evaluables: 206 de 545
Máxima desviación de cero: 1.857174

Debería ser del orden de 1e-15, no mayor.


## Tareas

1. Imprime por separado `panel.groupby('nr')['lwage'].mean()` y observa **su índice**.
   Compáralo con el índice de `panel['lwage']`. ¿Sobre qué se están alineando las dos
   Series al restarse?
2. Pídele a un asistente de IA la forma correcta de restar una media de grupo
   conservando la forma del *DataFrame* original.
3. Corrige y verifica con la prueba de arriba: la desviación máxima debe caer a ~1e-16.
4. Compara los coeficientes antes y después. ¿Por cuánto se equivocaba el efecto
   sindical? ¿Y la prima por matrimonio?

**La moraleja, que es la más importante de las cuatro actividades:** la pregunta de
verificación no es *«¿corrió?»* sino *«¿tiene la propiedad matemática que debía
tener?»*. Casi todo objeto econométrico trae consigo una identidad comprobable en dos
líneas de código.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las otras actividades de depuración.

In [5]:
# Tu corrección aquí.
#
# Pista: existe un método de groupby que devuelve el resultado con la forma
# del DataFrame original, repitiendo el valor del grupo en todos sus renglones.